<a href="https://colab.research.google.com/github/ksmorozov7/Portfolio/blob/main/Anti%E2%80%91fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Заливка данных

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import sqlite3

In [ ]:
sqlite3.sqlite_version

'3.45.1'

In [ ]:
! ls '/content/drive/My Drive/data'

Anti_fraud.db  german_credit_augmented.csv		 german_credit.db
emp_dept.db    german_credit_augmented_transactions.csv


In [ ]:
con = sqlite3.connect('/content/drive/My Drive/data/Anti_fraud.db')

## Функция для работы select

In [ ]:
def select(sql):
  return pd.read_sql(sql,con)

## Функция для работы
- DDL — Data Definition Language (меняет структуру);
- DML — Data Manipulation Language (меняет данные);
- DCL/TCL — управление транзакциями (реже)

In [ ]:
def execute(sql):
    cur = con.cursor()
    cur.execute(sql)
    con.commit()

## Матрица ошибок (Confusion Matrix):
• TP (True Positive): система заблокировала фрод.

• FP (False Positive): система заблокировала честного клиента (ложное срабатывание).

• FN (False Negative): система пропустила фрод (пропуск фрода).

• TN (True Negative): система правильно пропустила честного клиента.

## Таблицы

In [ ]:
df_transactions = pd.DataFrame({'amount':[100,200,300,400],
                  'is_blocked':[1,1,1,0],
                  'is_fraud':[1,1,0,0],
                  'date':['17-DEC-1980','20-FEB-1981','22-FEB-1981','02-APR-1981']})

In [ ]:
df_transactions

,amount,is_blocked,is_fraud,date
0,100,1,1,17-DEC-1980
1,200,1,1,20-FEB-1981
2,300,1,0,22-FEB-1981
3,400,0,0,02-APR-1981


In [ ]:
df_transactions.dtypes

,0
amount,int64
is_blocked,int64
is_fraud,int64
date,object


In [ ]:
df_transactions['date'] = pd.to_datetime(df_transactions['date'],format='%d-%b-%Y')

In [ ]:
df_transactions.dtypes

,0
amount,int64
is_blocked,int64
is_fraud,int64
date,datetime64[ns]


In [ ]:
df_transactions.to_sql('Transactions',con,index=False,if_exists='replace')

4

In [ ]:
sql = '''select * from Transactions t'''

In [ ]:
select(sql)

,amount,is_blocked,is_fraud,date
0,100,1,1,1980-12-17 00:00:00
1,200,1,1,1981-02-20 00:00:00
2,300,1,0,1981-02-22 00:00:00
3,400,0,0,1981-04-02 00:00:00


## Бизнес-метрики: AFPR (коэффициент ложных срабатываний) и Ущерб
AFPR (Anti-Fraud False Positive Ratio) — сколько честных пользователей мы блокируем на 1 пойманного мошенника. Идеально, когда показатель ниже 3:1 или 5:1 (зависит от ниши)

In [ ]:
sql = ''' WITH stats AS (
    SELECT
        SUM(CASE WHEN is_blocked = 1 AND is_fraud = 1 THEN 1 ELSE 0 END) * 1.0 AS TP_count,
        SUM(CASE WHEN is_blocked = 1 AND is_fraud = 0 THEN 1 ELSE 0 END) * 1.0 AS FP_count,

        SUM(CASE WHEN is_blocked = 0 AND is_fraud = 1 THEN amount ELSE 0 END) AS financial_loss_FN,
        SUM(CASE WHEN is_blocked = 1 AND is_fraud = 0 THEN amount ELSE 0 END) AS blocked_turnover_FP
    FROM Transactions
)
SELECT
    -- AFPR (Отношение FP к TP)
    (FP_count / NULLIF(TP_count, 0)) AS AFPR,

    -- Прямой убыток от пропущенного фрода
    financial_loss_FN AS Fraud_Loss_Money,

    -- Оборот честных клиентов, который заморозил (упущенная прибыль)
    blocked_turnover_FP AS Blocked_Good_Money
FROM stats '''


In [ ]:
select(sql)

,AFPR,Fraud_Loss_Money,Blocked_Good_Money
0,0.5,0,300


## AFR, FPR, FDR и SAR

In [ ]:
sql_metrics = '''
WITH matrix AS (
    SELECT
        SUM(CASE WHEN is_blocked = 1 AND is_fraud = 1 THEN 1 ELSE 0 END) * 1.0 AS TP,
        SUM(CASE WHEN is_blocked = 1 AND is_fraud = 0 THEN 1 ELSE 0 END) * 1.0 AS FP,
        SUM(CASE WHEN is_blocked = 0 AND is_fraud = 1 THEN 1 ELSE 0 END) * 1.0 AS FN,
        SUM(CASE WHEN is_blocked = 0 AND is_fraud = 0 THEN 1 ELSE 0 END) * 1.0 AS TN
    FROM Transactions
)
SELECT
    -- FPR: Какой % честных клиентов заблокировали? (Цель: как можно ближе к 0%)
    (FP / (FP + TN)) * 100 AS FPR_Percent,

    -- FDR: Какой % среди всех блокировок сделан ошибочно?
    (FP / (TP + FP)) * 100 AS FDR_Percent,

    -- AFR: Какова общая доля фрода в потоке транзакций?
    ((TP + FN) / (TP + FP + FN + TN)) * 100 AS AFR_Percent,

    -- SAR: Какой % от всех транзакций антифрод останавливает?
    ((TP + FP) / (TP + FP + FN + TN)) * 100 AS SAR_Percent
FROM matrix;
'''


In [ ]:
select(sql_metrics)

,FPR_Percent,FDR_Percent,AFR_Percent,SAR_Percent
0,50.0,33.333333,50.0,75.0


• FPR_Percent равен 50% (было всего 2 честных клиента, одного из них на 300 рублей заблокировали).

• FDR_Percent равен 33.3% (система сделала 3 блокировки, одна из них ошибочна: 1 из 3).

• AFR_Percent равен 50% (ровно половина транзакций в базе — фрод: 2 из 4).

• SAR_Percent равен 75% (система заблокировала 3 транзакции из 4).